# An Interpreter for a Simple Programming Language

In this notebook we develop an interpreter for a small programming language.  We will utilize the scanner generator
`ocamllex` and the parser generator `Menhir` to define our language rules and to generate a parse tree, which is
then converted into an abstract syntax tree.  (The Python version of this notebook uses the library `lark`.)

In [ ]:
#use "../lib/nb.ml";;
load_lib "parser_generator.ml";;
open Parse_tree

## Specification of the Grammar

Below is the grammar for our simple `C`-like language.  As explained in the notebook
`../Chapter-07/01-Calculator.ipynb`, the scanner and the parser are specified separately, and the semantic actions of
the grammar build a parse tree in the style of `lark`: a rule with an alias like `-> if_stmnt` creates a node with
the name `if_stmnt`, while a rule without an alias simply returns its only child.

Our programming language allows both single-line (`//`) and multi-line (`/* */`) comments.  The scanner ignores
them alongside standard whitespace.  A multi-line comment is skipped by the second rule `comment` of the scanner,
which reads characters until it finds the string `*/`.

The scanner specification is written as a *quoted string* `{|...|}`, so that backslashes do not have to be escaped.

In [ ]:
let sl_lexer = {|
{
  open Sl_parser
}

let digit = ['0'-'9']
let cname = ['_' 'a'-'z' 'A'-'Z'] ['_' 'a'-'z' 'A'-'Z' '0'-'9']*

rule token = parse
  | [' ' '\t' '\r' '\012']+ { token lexbuf }                     (* ignore whitespace *)
  | '\n'                    { Lexing.new_line lexbuf; token lexbuf }
  | "//" [^ '\n']*          { token lexbuf }                     (* CPP_COMMENT *)
  | "/*"                    { comment lexbuf; token lexbuf }     (* C_COMMENT *)
  | "if"                    { IF }
  | "while"                 { WHILE }
  | ":="                    { ASSIGN }
  | "=="                    { EQ }
  | "!="                    { NE }
  | "<="                    { LE }
  | ">="                    { GE }
  | '<'                     { LT }
  | '>'                     { GT }
  | '+'                     { PLUS }
  | '-'                     { MINUS }
  | '*'                     { STAR }
  | '/'                     { SLASH }
  | '%'                     { PERCENT }
  | '('                     { LPAR }
  | ')'                     { RPAR }
  | '{'                     { LBRACE }
  | '}'                     { RBRACE }
  | ';'                     { SEMICOLON }
  | ','                     { COMMA }
  | digit+                  { NUMBER (Parse_tree.token "NUMBER" lexbuf) }
  | cname                   { IDENTIFIER (Parse_tree.token "IDENTIFIER" lexbuf) }
  | eof                     { EOF }
  | _                       { raise (Parse_tree.unexpected lexbuf ("No terminal matches " ^ Lexing.lexeme lexbuf)) }

and comment = parse
  | "*/"                    { () }
  | '\n'                    { Lexing.new_line lexbuf; comment lexbuf }
  | eof                     { raise (Parse_tree.unexpected lexbuf "unterminated comment") }
  | _                       { comment lexbuf }
|}

`lark` supports the operators `*`, `+`, and `?` of *EBNF*.  The grammar below describes lists with recursive rules
instead: `stmnt_list` is a possibly empty list of statements, like `stmnt*` in `lark`, and `expr_seq` is a non-empty
list of expressions that are separated by commas, so `expr_list` corresponds to `(expr ("," expr)*)?`.  These rules
return OCaml lists of parse trees, so their type is declared as `Parse_tree.tree list`.

In [ ]:
let sl_grammar = {|
%{
  open Parse_tree
%}

%token <Parse_tree.tok> IDENTIFIER NUMBER
%token IF WHILE ASSIGN EQ NE LE GE LT GT PLUS MINUS STAR SLASH PERCENT
%token LPAR RPAR LBRACE RBRACE SEMICOLON COMMA EOF

%start <Parse_tree.tree> start
%type  <Parse_tree.tree> program stmnt bool_expr expr product factor expr_list
%type  <Parse_tree.tree list> stmnt_list expr_seq

%%

start: p = program EOF                                 { p }

program: l = stmnt_list                                { Tree ("prog", l) }

stmnt_list: /* empty */                                { [] }
          | s = stmnt l = stmnt_list                   { s :: l }

stmnt: IF LPAR b = bool_expr RPAR s = stmnt            { Tree ("if_stmnt", [b; s]) }
     | WHILE LPAR b = bool_expr RPAR s = stmnt         { Tree ("while_stmnt", [b; s]) }
     | LBRACE p = program RBRACE                       { Tree ("block", [p]) }
     | id = IDENTIFIER ASSIGN e = expr SEMICOLON       { Tree ("assign", [Token id; e]) }
     | e = expr SEMICOLON                              { Tree ("expr_stmnt", [e]) }

bool_expr: l = expr EQ r = expr                        { Tree ("eq", [l; r]) }
         | l = expr NE r = expr                        { Tree ("ne", [l; r]) }
         | l = expr LE r = expr                        { Tree ("le", [l; r]) }
         | l = expr GE r = expr                        { Tree ("ge", [l; r]) }
         | l = expr LT r = expr                        { Tree ("lt", [l; r]) }
         | l = expr GT r = expr                        { Tree ("gt", [l; r]) }

expr: e = expr PLUS p = product                        { Tree ("add", [e; p]) }
    | e = expr MINUS p = product                       { Tree ("sub", [e; p]) }
    | p = product                                      { p }

product: p = product STAR f = factor                   { Tree ("mul", [p; f]) }
       | p = product SLASH f = factor                  { Tree ("div", [p; f]) }
       | p = product PERCENT f = factor                { Tree ("mod", [p; f]) }
       | f = factor                                    { f }

factor: LPAR e = expr RPAR                             { e }
      | n = NUMBER                                     { Tree ("number", [Token n]) }
      | id = IDENTIFIER                                { Tree ("var", [Token id]) }
      | id = IDENTIFIER LPAR l = expr_list RPAR        { Tree ("fct_call", [Token id; l]) }

expr_list: /* empty */                                 { Tree ("explist", []) }
         | l = expr_seq                                { Tree ("explist", l) }

expr_seq: e = expr                                     { [e] }
        | e = expr COMMA l = expr_seq                  { e :: l }
|}

We generate the scanner and the parser.  `menhir --lalr` creates a *Look-Ahead Left-to-Right* (LALR) parser.

In [ ]:
generate_parser ~name:"sl" ~lexer:sl_lexer ~grammar:sl_grammar

## Generating the Abstract Syntax Tree (ADT)

To maintain a clean functional boundary, we walk the parse tree generated by the parser and map it to an abstract
data type.  In Python, this ADT consists of nested tuples like `(':=', 'x', ('+', 'y', 1))`.  In OCaml, we define the
variant types `expr` for arithmetic expressions, `bool_expr` for the tests of `if` and `while`, and `stmnt` for
statements.

**Type safety:** In the Python version, a test like `x < y` and an arithmetic expression like `x + y` are both nested
tuples, so nothing prevents a test from being used as an argument of `+`.  In OCaml, they have different types.
Furthermore, the compiler checks that the functions `execute`, `evaluate_bool`, and `evaluate` handle all cases,
so the final cases `assert False, f'{stmnt} unexpected'` of the Python version are not needed.

In [ ]:
type arith_op = Add | Sub | Mul | Div | Mod
type cmp_op   = Eq | Ne | Le | Ge | Lt | Gt

type expr =
  | Num   of int
  | Var   of string
  | Call  of string * expr list           (* ('call', f_name, arg1, ..., argn) *)
  | Arith of arith_op * expr * expr

type bool_expr = Cmp of cmp_op * expr * expr

type stmnt =
  | Prog   of stmnt list                  (* ('.', stmnt1, ..., stmntn)  *)
  | Assign of string * expr               (* (':=', var, expr)           *)
  | Expr   of expr                        (* ('expr', expr)              *)
  | If     of bool_expr * stmnt           (* ('if', test, stmnt)         *)
  | While  of bool_expr * stmnt           (* ('while', test, stmnt)      *)

`arith_map` and `cmp_map` map the names of the nodes of the parse tree to the operators.  They correspond to the
dictionary `SYMBOL_MAP` of the Python version.  The functions `arith_symbol` and `cmp_symbol` return the symbols
that are used for the operators in the Python version.

In [ ]:
let arith_map : (string * arith_op) list = [ ("add", Add); ("sub", Sub); ("mul", Mul); ("div", Div); ("mod", Mod) ]
let cmp_map   : (string * cmp_op) list   = [ ("eq", Eq); ("ne", Ne); ("le", Le); ("ge", Ge); ("lt", Lt); ("gt", Gt) ]

let arith_symbol (op : arith_op) : string =
  match op with Add -> "+" | Sub -> "-" | Mul -> "*" | Div -> "/" | Mod -> "%"

let cmp_symbol (op : cmp_op) : string =
  match op with Eq -> "==" | Ne -> "!=" | Le -> "<=" | Ge -> ">=" | Lt -> "<" | Gt -> ">"

`ast_to_stmnt` takes a node of the parse tree and transforms it into a statement.  It uses the functions `ast_to_expr`
and `ast_to_bool`, which transform nodes into expressions and tests.  A block `{ ... }` is represented by the
statement list that it contains.

In [ ]:
let rec ast_to_expr (node : tree) : expr =
  match node with
  | Tree ("number", [ Token t ]) -> Num (int_of_string t.value)
  | Tree ("var", [ Token t ]) -> Var t.value
  | Tree ("fct_call", [ Token f; Tree ("explist", args) ]) -> Call (f.value, List.map ast_to_expr args)
  | Tree (name, [ lhs; rhs ]) when List.mem_assoc name arith_map ->
      Arith (List.assoc name arith_map, ast_to_expr lhs, ast_to_expr rhs)
  | _ -> invalid_arg ("Unknown syntax tree node: " ^ Parse_tree.to_string node)

let ast_to_bool (node : tree) : bool_expr =
  match node with
  | Tree (name, [ lhs; rhs ]) when List.mem_assoc name cmp_map ->
      Cmp (List.assoc name cmp_map, ast_to_expr lhs, ast_to_expr rhs)
  | _ -> invalid_arg ("Unknown syntax tree node: " ^ Parse_tree.to_string node)

let rec ast_to_stmnt (node : tree) : stmnt =
  match node with
  | Tree ("prog", stmnts) -> Prog (List.map ast_to_stmnt stmnts)
  | Tree ("block", [ program ]) -> ast_to_stmnt program
  | Tree ("assign", [ Token var; e ]) -> Assign (var.value, ast_to_expr e)
  | Tree ("expr_stmnt", [ e ]) -> Expr (ast_to_expr e)
  | Tree ("if_stmnt", [ test; s ]) -> If (ast_to_bool test, ast_to_stmnt s)
  | Tree ("while_stmnt", [ test; s ]) -> While (ast_to_bool test, ast_to_stmnt s)
  | _ -> invalid_arg ("Unknown syntax tree node: " ^ Parse_tree.to_string node)

The external helper file `AST2Dot.ipynb` is used to represent our abstract syntax tree graphically.  It draws nested
tuples of the type `nested`.  The functions below convert an abstract syntax tree into a nested tuple that has the
same structure as the nested tuples of the Python version.

In [ ]:
run_notebook "AST2Dot.ipynb"
;;
let rec expr_to_nested (e : expr) : nested =
  match e with
  | Num n -> Int n
  | Var v -> Str v
  | Call (f, args) -> Tuple ("call", Str f :: List.map expr_to_nested args)
  | Arith (op, lhs, rhs) -> Tuple (arith_symbol op, [ expr_to_nested lhs; expr_to_nested rhs ])

let bool_to_nested (Cmp (op, lhs, rhs) : bool_expr) : nested =
  Tuple (cmp_symbol op, [ expr_to_nested lhs; expr_to_nested rhs ])

let rec to_nested (s : stmnt) : nested =
  match s with
  | Prog sl -> Tuple (".", List.map to_nested sl)
  | Assign (var, e) -> Tuple (":=", [ Str var; expr_to_nested e ])
  | Expr e -> Tuple ("expr", [ expr_to_nested e ])
  | If (test, s) -> Tuple ("if", [ bool_to_nested test; to_nested s ])
  | While (test, s) -> Tuple ("while", [ bool_to_nested test; to_nested s ])

The function `parse` takes a `file_name` as its sole argument. The file is read, passed through the parser, mapped to
our abstract syntax tree, and visualized via `Graphviz`.  The abstract syntax tree is printed as a nested tuple in the
syntax of Python.

In [ ]:
let parse (file_name : string) : Graphviz.t =
  let program = read_file file_name in
  print_endline program;
  let tree = parse_string Sl_parser.start Sl_lexer.token program in
  let ast  = ast_to_stmnt tree in
  print_endline (nested_to_string (to_nested ast));
  tuple2dot (to_nested ast)

In [ ]:
Graphviz.display (parse "sum.sl")

The values of our language are numbers.  As in Python, a number is either an integer or a floating point number:
the operator `/` always returns a floating point number, e.g. `7 / 2` is `3.5`, and the operator `%` returns a result
that has the same sign as its second argument.  The type `number` and the following functions implement this
arithmetic.

**Note:** Python's integers can be arbitrarily large, while OCaml's integers have 63 bits.  Hence, the factorial of
a number bigger than 20 can not be represented as an integer of type `int`.

In [ ]:
type number = I of int | F of float

let to_float (n : number) : float = match n with I k -> float_of_int k | F x -> x

let arith (int_op : int -> int -> int) (float_op : float -> float -> float) (a : number) (b : number) : number =
  match a, b with
  | I x, I y -> I (int_op x y)
  | _ -> F (float_op (to_float a) (to_float b))

let python_mod (x : int) (y : int) : int = ((x mod y) + y) mod y

let python_fmod (x : float) (y : float) : float = x -. y *. Float.floor (x /. y)

let num_to_string (n : number) : string = match n with I k -> string_of_int k | F x -> float_repr x

The function `execute_tuple` loops through statements linearly. The hash table `values` is the continuous state that
gets updated as assignments occur.  Since the functions `execute_tuple`, `execute`, `evaluate_bool`, and `evaluate`
call each other, they are defined together in the next cells using forward references.

In [ ]:
let not_yet_defined _ = failwith "not yet defined"

let execute_ref       : (stmnt -> (string, number) Hashtbl.t -> unit) ref     = ref not_yet_defined
let evaluate_bool_ref : (bool_expr -> (string, number) Hashtbl.t -> bool) ref = ref not_yet_defined
let evaluate_ref      : (expr -> (string, number) Hashtbl.t -> number) ref    = ref not_yet_defined

let execute_tuple (statementList : stmnt list) (values : (string, number) Hashtbl.t) : unit =
  List.iter (fun stmnt -> !execute_ref stmnt values) statementList

The function `execute` matches recursively against the shape of the statement, triggering evaluation or recursive
execution sequences.  The `while` loop of Python is replaced by a recursive function `loop`.

In [ ]:
let rec execute (stmnt : stmnt) (values : (string, number) Hashtbl.t) : unit =
  match stmnt with
  | Prog sl -> execute_tuple sl values
  | Assign (var, value) -> Hashtbl.replace values var (!evaluate_ref value values)
  | Expr expr -> ignore (!evaluate_ref expr values)
  | If (test, stmnt) -> if !evaluate_bool_ref test values then execute stmnt values
  | While (test, stmnt) ->
      let rec loop () = if !evaluate_bool_ref test values then (execute stmnt values; loop ()) in
      loop ()
;;
execute_ref := execute

The function `evaluate_bool` executes conditional logical statements returning a pure boolean output.
Numbers are compared by their values, so `1 == 1.0` is true, as in Python.

In [ ]:
let evaluate_bool (expr : bool_expr) (values : (string, number) Hashtbl.t) : bool =
  match expr with
  | Cmp (op, lhs, rhs) -> (
      let l = to_float (!evaluate_ref lhs values) in
      let r = to_float (!evaluate_ref rhs values) in
      match op with
      | Eq -> l = r
      | Ne -> l <> r
      | Le -> l <= r
      | Ge -> l >= r
      | Lt -> l < r
      | Gt -> l > r)
;;
evaluate_bool_ref := evaluate_bool

The function `evaluate` calculates specific numerical values given either primitives (numbers, variables) or compound
arithmetic operations.  We utilize OCaml's `match` to guarantee exhaustive mapping of our ADT types.  The function
`read` reads a number with the function `input` from the file `../lib/nb.ml`, since the OCaml kernel can not read
from the keyboard.

In [ ]:
let rec evaluate (expr : expr) (values : (string, number) Hashtbl.t) : number =
  match expr with
  | Num n -> I n
  | Var var -> Hashtbl.find values var
  | Call ("read", []) -> I (int_of_string (input "Please enter a natural number: "))
  | Call ("print", [ expr ]) ->
      print_endline (num_to_string (evaluate expr values));
      I 0
  | Call (f, _) -> failwith (Printf.sprintf "function %s is unknown" f)
  | Arith (op, lhs, rhs) -> (
      let l = evaluate lhs values in
      let r = evaluate rhs values in
      match op with
      | Add -> arith ( + ) ( +. ) l r
      | Sub -> arith ( - ) ( -. ) l r
      | Mul -> arith ( * ) ( *. ) l r
      | Div -> F (to_float l /. to_float r)
      | Mod -> arith python_mod python_fmod l r)
;;
evaluate_ref := evaluate

In [ ]:
cat "sum.sl"

Finally, we bring the whole pipeline together. `main` will open the program file, parse it fully, build our abstract
syntax tree, and evaluate it under an isolated state.  The program calls `read()`, so we first tell the function
`input` which number the user enters.

In [ ]:
let main (file : string) : unit =
  let program = read_file file in
  let tree = parse_string Sl_parser.start Sl_lexer.token program in
  let ast  = ast_to_stmnt tree in
  Graphviz.display (tuple2dot (to_nested ast));
  let values = Hashtbl.create 16 in
  execute ast values

In [ ]:
simulate_input [ "10" ];;
main "sum.sl"

In [ ]:
cat "factorial.sl"

In [ ]:
simulate_input [ "10" ];;
main "factorial.sl"